In [32]:
import numpy as np
import tensorflow as tf
import pickle

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [33]:
# Load IMDB RAW TEXT


num_words = 10000
max_len = 500

# Load raw dataset (not pre-encoded)
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=num_words)


In [34]:
word_index = imdb.get_word_index()

In [35]:
# Convert integer sequences back to words
reverse_word_index = {v: k for k, v in word_index.items()}

In [36]:
def decode_review(encoded_review):
    return " ".join([reverse_word_index.get(i - 3, "?") for i in encoded_review])

X_train_text = [decode_review(review) for review in X_train]
X_test_text = [decode_review(review) for review in X_test]


In [37]:
# Tokenizer 

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

In [38]:
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [39]:
# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [40]:
# Building the  Model 

model = Sequential()
model.add(Embedding(num_words, 128, input_length=max_len)) ## Embedding layer
model.add(SimpleRNN(128, activation="relu")) ## Simple RNN layer
model.add(Dense(1, activation="sigmoid")) ## Output layer

In [41]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [42]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [43]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 126s 196ms/step - accuracy: 0.6023 - loss: 5276650.5000 - val_accuracy: 0.6754 - val_loss: 0.5893
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 135s 184ms/step - accuracy: 0.7844 - loss: 0.6061 - val_accuracy: 0.7836 - val_loss: 0.4428
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 145s 232ms/step - accuracy: 0.8847 - loss: 0.2797 - val_accuracy: 0.8240 - val_loss: 0.3839
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 192s 216ms/step - accuracy: 0.9320 - loss: 0.1757 - val_accuracy: 0.8366 - val_loss: 0.4065
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 143s 229ms/step - accuracy: 0.9584 - loss: 0.1170 - val_accuracy: 0.8326 - val_loss: 0.4436
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 116s 185ms/step - accuracy: 0.9701 - loss: 0.0852 - val_accuracy: 0.8342 - val_loss: 0.5006
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 155s 205ms/step - accuracy: 0.9753 - loss: 0.0786 - val_accuracy: 0.8284 - val_loss: 0.5571
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 142s 228ms/step - accuracy: 0.

In [44]:
model.save_weights("model.weights.h5")

print("Training complete. Model and tokenizer saved.")

Training complete. Model and tokenizer saved.
